In [1]:
load("fun.sage")

In [2]:
# N number of qubits 

N = 20

data = {}

for n in range(8,N):

    k=floor(n/2)

    A,b,c=Standard_form(n,k) # constraints in the form of A

    primal = MixedIntegerLinearProgram(
        maximization=False,
        base_ring=QQ,
        solver="PPL"
    )
    
    x = primal.new_variable(real=True) # primal variables a' vector in the paper
    
    primal.set_objective(
        sum(c[0,j]*x[j] for j in range(n+1))
    ) # objective function c*x
    
    for i in range(A.nrows()):
        primal.add_constraint(
            sum(A[i,j]*x[j] for j in range(n+1)) >= b[i,0]
        ) # Constraints A*x >= b
    
    for j in range(n+1):
        primal.add_constraint(x[j] >= 0) # Positivity of x>=0 (This condition is redundant, but by convention,
                                         # the standard form has positive variables)
    
    primal.solve() # Solve primal
    
    dual = MixedIntegerLinearProgram(
        maximization=True,
        base_ring=QQ,
        solver="PPL"
    )
    
    y = dual.new_variable(real=True) # dual variables
    
    dual.set_objective(
        sum(b[i,0]*y[i] for i in  range(A.nrows()))
    ) # the objective function of the dual is b*y
    
    for j in range(n+1):
        dual.add_constraint(
            sum(A[i,j]*y[i] for i in range(A.nrows())) <= c[0,j]
        ) # y*A =< c is the dual contraints
    
    for j in range(A.nrows()):
        dual.add_constraint(y[j] >= 0) # y>=0 are positive
    
    dual.solve() # solve dual
    
    x_sol = matrix(QQ, [[primal.get_values(x[i]) for i in range(n+1)]]) # primal solution
    y_sol = matrix(QQ, [[dual.get_values(y[i]) for i in range(A.nrows())]]) # dual solution
    
    if A*x_sol.transpose() >= b and x_sol >= 0 and y_sol*A <= c and y_sol >= 0 and c*x_sol.transpose()==y_sol*b:
    
        """
        This condition checks if both solutions satisfy the respective conditions + strong duality, i.e.
    
        Primal            Dual
        
        min c*x^T           max b*y
        s.t. Ax >= b      s.t. yB <= c
             x_i >= 0            y_i >= 0
    
        Strong duality: c*x^T = y*b
        
        """
        obj= c*x_sol.transpose()
    
        data[n]=[x_sol,y_sol,obj]

        # save(data, "TE_new.sobj")

In [3]:
data

{8: [[1 4 7 7 6 7 7 4 1],
  [   6    0    0   35    0    0    0    0    0    0 11/4    1  1/4    0    0    0    0    0    1    0    0    0    0    0    0    0    0    0 5/16  1/4 5/16],
  [6]],
 9: [[   1  9/2    9 21/2    9    9 21/2    9  9/2    1],
  [      9       0       0       0       0       0       0       0       0       0       0       0       4    11/8     3/8       0       0       0       0       0       0 315/256     1/2  35/256       0       0       0   3/256       0       0       0       0       0   5/256       0   5/256],
  [9]],
 10: [[   1    5 45/4   15 55/4   12 55/4   15 45/4    5    1],
  [     12       0       0     504       0       0       0       0       0       0       0       0       6       2     3/4       0       0       0       0       0       0       0     3/2       0     1/2       0       0       0       0       0       0       0       0       0 105/128   65/64 123/128   33/32],
  [12]],
 11: [[    1  11/2  55/4 165/8 165/8  33/2  33/2 165/8 165/8  55/

In [7]:
2^5

32